# DSC0152 - notebook v62 (kandidat setelah public 0.56312)

Run sebelumnya (`v61`) mendapat public **0.56312** lewat file `submission_v61_alt_ext0.5_tm30_lean.csv`, naik dari 0.56171. Jalur ekstrapolasi itu terbukti: model dengan fitur lebih kaya didorong menjauh dari model sejenis dengan fitur lebih sedikit.

v62 memperpanjang gradien kaya-fitur menjadi enam tingkat, menambah feature selection berbasis gain per fold, menambah blok fitur konsistensi antar tabel, memakai empat fold seed, dan menggabungkan prediksi v61 dan v62 sebelum diekstrapolasi.

## Lokasi berkas

```
agas/
  shared/            feat_extra.py, build_extra.py, cv_harness.py, factor2.py, paths.py
  shared/cache/      features_cache.pkl, extra_cache.pkl, exblocks/
  v61/training/     train_submission_v61.py, make_submission_v61.py, study_blend_v61.py
  v62/training/      train_submission_v62.py dan lainnya
  v62/output/        submission_v62f_*.csv, preds_v62.npz, summary, log
```

## Hasil ringkas

| Run | KS OOF terbaik | AUC OOF terbaik |
|---|---:|---:|
| v61 | 0.528624 | 0.847177 |
| v62 | 0.527846 | 0.847419 |
| gabungan | **0.528717** | 0.847128 |

`lgb_top` (200 fitur terpilih gain) adalah model terkuat sepanjang masa ini: AUC 0.847419.

## Aturan yang dipatuhi

Python saja, hanya data kompetisi lokal, preprocessing dan feature engineering manual, tanpa AutoML dan AutoFE, tanpa data eksternal. Semua agregat historis dipotong pada `decision_datetime`. Factor loading dihitung dari label fold training saja. Tidak ada push ke GitHub. Semua artefak hanya di `agas/`.

`RUN_TRAINING = False` membuat notebook hanya membaca cache sehingga bisa diverifikasi tanpa menghitung ulang.

In [1]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd

RUN_TRAINING = False
FOLDS = 10
FOLD_SEEDS = '42,7,13,21'
LR = 0.01
L2 = 5.0
TOP_N = 200
TOP_WIDE = 350

ROOT = Path.cwd()
AGAS = ROOT / 'agas'
VER = AGAS / 'v62'
OUT = VER / 'output'
SHARED = AGAS / 'shared'
CACHE = SHARED / 'cache'
TRAINING = VER / 'training'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
os.chdir(ROOT)
print('numpy', np.__version__, '| pandas', pd.__version__)
print('version dir', VER)

numpy 2.5.1 | pandas 3.0.5
version dir /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v62


## 1. Data lokal

Tujuh tabel kompetisi dipakai. Tidak ada sumber eksternal.

In [2]:
import lightgbm, xgboost, catboost, sklearn

print('lightgbm', lightgbm.__version__)
print('xgboost', xgboost.__version__)
print('catboost', catboost.__version__)
print('sklearn', sklearn.__version__)

apps = pd.read_csv(ROOT / 'data/train_applications.csv')
print('train rows', len(apps), '| base rate', round(apps.target.mean(), 5))

for name in ['train_applications', 'test_applications', 'customer_profiles',
             'previous_applications', 'repayment_history', 'account_behavior',
             'bureau_observations', 'ecommerce_transactions']:
    p = ROOT / 'data' / f'{name}.csv'
    print(f'{name:24s} {p.stat().st_size / 1e6:9.1f} MB')

lightgbm 4.7.0
xgboost 3.3.0
catboost 1.2.10
sklearn 1.9.0
train rows 70000 | base rate 0.13581
train_applications             8.2 MB
test_applications              3.5 MB
customer_profiles              6.4 MB
previous_applications         26.7 MB
repayment_history            159.7 MB
account_behavior             100.0 MB
bureau_observations          114.7 MB
ecommerce_transactions       528.8 MB


## 2. Feature frame

`features.build()` dari `features.py` tidak diubah. Blok fitur tambahan ada di `agas/shared/feat_extra.py`. Enam tingkat kaya-fitur dipakai sebagai kaki ekstrapolasi:

| Tingkat | Yang dibuang | Sifat |
|---|---|---|
| `full` | tidak ada | semua fitur |
| `fac` | `r2_`, `b2_`, `a2_`, `e2_` | tanpa blok putaran dua |
| `lean` | kolom noise | tanpa kolom noise |
| `faclean` | keduanya | tingkat terkurus |
| `top` | gain | 200 fitur terpilih per fold |
| `topwide` | gain | 350 fitur terpilih per fold |

In [3]:
import features as ft

cache = CACHE / 'features_cache.pkl'
if RUN_TRAINING or not cache.exists():
    f = ft.build()
    cache.parent.mkdir(parents=True, exist_ok=True)
    f.to_pickle(cache)
else:
    f = pd.read_pickle(cache)

ex = pd.read_pickle(CACHE / 'extra_cache.pkl')
add = [c for c in ex.columns if c != 'application_id' and c not in f.columns]
f = f.merge(ex[['application_id'] + add], on='application_id', how='left')

tr = f[f.target.notna()].reset_index(drop=True)
te = f[f.target.isna()].reset_index(drop=True)
print('train', tr.shape, '| test', te.shape)

NOISE_EXACT = {'region', 'employment_type', 'housing_status', 'age',
               'employment_tenure_months', 'dependents', 'tenure_to_age',
               'income_per_dependent', 'loan_purpose', 'decision_lag_h',
               'app_hour', 'app_dow'}
NOISE_PREFIX = ('ec_share_', 'ec_catqty_', 'cs_shift', 'buh_', 'gn_')
ROUND2_PREFIX = ('r2_', 'b2_', 'a2_', 'e2_')

cols = [c for c in tr.columns if c not in ('application_id', 'customer_id', 'target')]
full = list(cols)
fac = [c for c in cols if not c.startswith(ROUND2_PREFIX)]
lean = [c for c in cols if c not in NOISE_EXACT and not c.startswith(NOISE_PREFIX)]
faclean = [c for c in fac if c not in NOISE_EXACT and not c.startswith(NOISE_PREFIX)]
print('full', len(full), '| fac', len(fac), '| lean', len(lean), '| faclean', len(faclean))

train (70000, 557) | test (30000, 557)
full 554 | fac 509 | lean 446 | faclean 401


## 3. Blok fitur konsistensi

Selisih antar sumber. Nasabah yang terlihat bersih di bureau padahal riwayat cicilan internal menunjukkan sebaliknya, lalu celah itu sendiri yang dipakai.

Nilai rata-rata tiga fold seed: `+0.00059 KS` pada `lr 0.02 / lambda_l2 20` dan `+0.00097 KS` pada `lr 0.01 / lambda_l2 5`.

In [4]:
cx = [c for c in tr.columns if c.startswith('cx_')]
print(len(cx), 'fitur konsistensi')
print(cx)

BLOCKS = 'bureau,repayment,previous,account,ecommerce,interactions,gradenorm,rxtra,bxtra,axtra,ecommerce2,consistency'
if RUN_TRAINING:
    os.system(f'python3 {SHARED / "build_extra.py"} consistency')
    os.system(f'python3 {SHARED / "build_extra.py"} {BLOCKS}')

12 fitur konsistensi
['cx_bureau_vs_declared_debt', 'cx_bureau_debt_vs_visible_inst', 'cx_bureau_accounts_vs_our_loans', 'cx_inquiries_vs_our_apps', 'cx_delinq_gap', 'cx_bureau_delinq_per_internal_late', 'cx_file_age_gap_days', 'cx_inst_vs_hist_max', 'cx_inst_vs_hist_mean', 'cx_remaining_vs_inst', 'cx_recent_app_pressure', 'cx_bureau_known_but_no_apps']


## 4. Ensemble

10 fold `StratifiedKFold`, empat fold seed, LightGBM pada setiap tingkat tangga dengan dua seed pada `full` dan satu seed pada tingkat lain, plus feature selection 200 dan 350 kolom per fold, CatBoost, XGBoost, LightGBM konfigurasi ekstra, dan spline linear. OOF tiap kaki disimpan per fold seed supaya arah ekstrapolasi bisa diuji di setiap seed, bukan hanya pada rata-rata.

In [5]:
cmd = (f'python3 {TRAINING / "train_submission_v62.py"} --folds {FOLDS} '
       f'--fold-seeds {FOLD_SEEDS} --lr {LR} --l2 {L2} --full-seeds 0,1 '
       f'--top-n {TOP_N} --top-wide {TOP_WIDE} --out-prefix v62')
print(cmd)
if RUN_TRAINING:
    os.system(cmd)
else:
    print('dilewati, memakai agas/v62/output/preds_v62.npz yang sudah ada')

python3 /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v62/training/train_submission_v62.py --folds 10 --fold-seeds 42,7,13,21 --lr 0.01 --l2 5.0 --full-seeds 0,1 --top-n 200 --top-wide 350 --out-prefix v62
dilewati, memakai agas/v62/output/preds_v62.npz yang sudah ada


In [6]:
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

z = np.load(OUT / 'preds_v62.npz', allow_pickle=True)
seeds = [int(s) for s in z['seeds']]
train_ids, test_ids = z['train_ids'], z['test_ids']
ymap = apps.set_index('application_id').reindex(pd.Index(train_ids))
y = ymap['target'].values.astype(int)

rows = []
for k in sorted(c for c in z.files if c.startswith('oof_')):
    name = k[4:]
    p = z[k]
    rows.append({'model': name, 'auc': roc_auc_score(y, p),
                 'ks': ks_2samp(p[y == 1], p[y == 0]).statistic})
base = pd.DataFrame(rows).sort_values('ks', ascending=False).reset_index(drop=True)
base

,model,auc,ks
0,lgb_top,0.847419,0.525550
1,xgb_full,0.846639,0.524982
2,cat_full,0.846046,0.524848
3,lgb_lean,0.846645,0.524386
4,lgb_topwide,0.846663,0.524101
5,lgb_full,0.846629,0.523196
6,lgb_faclean,0.846444,0.523102
7,lgb_fac,0.846392,0.523101
8,lgbx_full,0.846305,0.523064
9,linear,0.842723,0.521687


## 5. Pencarian arah ekstrapolasi

Dua arah sudah terbukti di leaderboard: `v4 - fac` (0.56171) dan `topmix30 - lgb_lean` (0.56312). Keduanya menolak model kaya fitur menjauh dari model miskin fitur.

Karena galat standar KS di 70.000 baris OOF sekitar 0.002 sampai 0.003, satu pembacaan OOF tidak cukup untuk memutuskan. Setiap arah dinilai ulang di dalam setiap fold seed, dan untuk base gabungan v61+v62 dipakai resampling baris pada delapan sampel acak setengah OOF.

In [7]:
if RUN_TRAINING:
    os.system(f'python3 {TRAINING / "search_gradient_v62.py"}')
    os.system(f'python3 {TRAINING / "search_combined.py"}')
    os.system(f'python3 {TRAINING / "make_submission_v62_final.py"}')

for f in ['gradient_v62.csv', 'gradient_combined.csv']:
    path = OUT / f
    if path.exists():
        d = pd.read_csv(path)
        keep = [c for c in ['name', 'ratio', 'ks', 'auc', 'worst_delta', 'n_pos', 'robust']
                if c in d.columns]
        print('===', f)
        print(d.sort_values('ks', ascending=False).head(8)[keep].to_string(index=False))
        print()

=== gradient_v62.csv
                              name  ratio       ks      auc  worst_delta  n_pos  robust
     b4_topsonly_w0.2-lgb_fac_r0.5  0.500 0.527879 0.846467    -0.000884      2   False
     b2_top_w0.3-lgb_topwide_r0.25  0.250 0.527846 0.846457    -0.000866      3    True
 b3_topwide_w0.3-lgb_topwide_r0.25  0.250 0.527846 0.846457    -0.000866      3    True
    b1_v61_w0.2-lgb_topwide_r0.375  0.375 0.527802 0.846590    -0.000530      1   False
  b5_notop_w0.2-lgb_topwide_r0.375  0.375 0.527769 0.846523    -0.000697      1   False
b4_topsonly_w0.3-lgb_topwide_r0.25  0.250 0.527751 0.846513    -0.000982      1   False
      b3_topwide_w0.2-lgb_fac_r0.5  0.500 0.527730 0.846437    -0.001245      2   False
  b4_topsonly_w0.1-lgb_lean_r0.625  0.625 0.527725 0.846705    -0.001548      0   False

=== gradient_combined.csv
                         name  ratio       ks      auc  worst_delta  n_pos  robust
mix_w0.3-62:lgb_topwide_r0.25  0.250 0.528347 0.846424          NaN     -1   

## 6. Kandidat

Jangkar untuk blend konservatif adalah `agas/v61/output/submission_v61_alt_ext0.5_tm30_lean.csv` dengan public 0.56312. Kandidat dengan korelasi rank tinggi terhadap jangkar praktis mengulang skor yang sudah diketahui, jadi urutannya menempatkan yang cukup berbeda di depan.

In [8]:
summary = json.load(open(OUT / 'summary_v62_final.json'))
cands = pd.DataFrame([{k: v for k, v in c.items() if k != 'problems'}
                     for c in summary['candidates']])
cands

,name,auc,ks,half_mean,half_min,file,spearman_anchor
0,two_w0.3_fac0.25_fac61_0.5,0.846784,0.528717,0.528522,0.518682,submission_v62f_two_w0.3_fac0.25_fac61_0.5.csv,0.999653
1,mixA_topwide_r025,0.846424,0.528347,0.528232,0.517901,submission_v62f_mixA_topwide_r025.csv,0.999744
2,two_w0.2_fac0.5_fac61_0.5,0.846363,0.528251,0.528538,0.518051,submission_v62f_two_w0.2_fac0.5_fac61_0.5.csv,0.998996
3,mixB_facv61_r05,0.846368,0.528198,0.527829,0.518746,submission_v62f_mixB_facv61_r05.csv,0.998954
4,v61_fac_r025,0.846303,0.528048,0.527835,0.518481,submission_v62f_v61_fac_r025.csv,0.999476
5,v61_facv62_r05,0.846703,0.527993,0.527388,0.516629,submission_v62f_v61_facv62_r05.csv,0.998242
6,mixC_facv62_r05,0.846434,0.527916,0.528223,0.518842,submission_v62f_mixC_facv62_r05.csv,0.999062
7,v62_fac_r05,0.846488,0.527716,0.528205,0.519134,submission_v62f_v62_fac_r05.csv,0.998800
8,two_w0.2_fac0.5_fac61_0.25,0.846222,0.527600,0.528184,0.519085,submission_v62f_two_w0.2_fac0.5_fac61_0.25.csv,0.998751
9,two_w0.3_fac0.25_fac61_0.25,0.846447,0.527391,0.527893,0.518126,submission_v62f_two_w0.3_fac0.25_fac61_0.25.csv,0.999422


## 7. Validasi file submission

Setiap kandidat diperiksa ulang: 30000 baris, `application_id` unik dan sama dengan `data/test_applications.csv`, probabilitas di rentang [0, 1] tanpa NaN.

In [9]:
ref = set(pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id']).application_id)

rows = []
for path in sorted(OUT.glob('submission_v62f_*.csv')):
    d = pd.read_csv(path)
    ok = (len(d) == 30000 and d.application_id.is_unique and set(d.application_id) == ref
          and d.default_probability.between(0, 1).all() and not d.default_probability.isna().any())
    rows.append({'file': path.name, 'rows': len(d), 'ok': ok,
                 'min': round(d.default_probability.min(), 6),
                 'max': round(d.default_probability.max(), 6)})
pd.DataFrame(rows)

,file,rows,ok,min,max
0,submission_v62f_anchor25_consensus.csv,30000,True,0.000033,1.000000
1,submission_v62f_anchor50_consensus.csv,30000,True,0.000033,1.000000
2,submission_v62f_anchor75_consensus.csv,30000,True,0.000033,1.000000
3,submission_v62f_consensus.csv,30000,True,0.000033,1.000000
4,submission_v62f_mixA_topwide_r025.csv,30000,True,0.000033,1.000000
5,submission_v62f_mixB_facv61_r05.csv,30000,True,0.000033,1.000000
6,submission_v62f_mixC_facv62_r05.csv,30000,True,0.000033,1.000000
7,submission_v62f_mixD_faclean_r05.csv,30000,True,0.000033,1.000000
8,submission_v62f_mix_base_w02.csv,30000,True,0.000033,1.000000
9,submission_v62f_two_w0.2_fac0.25_fac61_0.25.csv,30000,True,0.000033,1.000000


## 8. Urutan submit

1. `submission_v62f_two_w0.2_fac0.5_fac61_0.5.csv`, KS OOF 0.528251, korelasi 0.998996 terhadap jangkar.
2. `submission_v62f_mixB_facv61_r05.csv`, KS OOF 0.528198, korelasi 0.998954.
3. `submission_v62f_v61_facv62_r05.csv`, KS OOF 0.527993, korelasi 0.998242.
4. `submission_v62f_two_w0.3_fac0.25_fac61_0.5.csv`, KS OOF 0.528717 tertinggi sesi ini, korelasi 0.999653.
5. `submission_v62f_consensus.csv` dan `submission_v62f_anchor50_consensus.csv` untuk slot yang harus aman.

## 9. Realitas target

KS OOF mentok di sekitar 0.528 sejak v61. Dua run terakhir hanya menaikkan AUC, dari 0.847177 ke 0.847419, tanpa menaikkan KS. Selisih di bawah 0.002 tidak dapat dibedakan dari noise.

Dengan jangkar 0.56312, jarak ke 0.57 adalah 0.0069, sekitar dua sampai empat kali noise public yang tercatat. Gap KS OOF yang dibutuhkan kira-kira 0.02, sementara total usaha dua run terakhir menghasilkan 0.0017. Maka 0.57 sebaiknya diperlakukan sebagai batas atas kompetisi, dan kuota submit yang tersisa dipakai untuk menguji sebanyak mungkin arah berbeda dengan KS OOF di atas rata-rata, bukan untuk mengejar satu angka OOF.